# Load and process E. coli data

In [1]:
import numpy as np
import pandas as pd
from Bio import AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load and check `.xmfa` data

In [2]:
def read_xmfa_biopython(file_path):
    blocks = list(AlignIO.parse(file_path, "mauve"))
    return blocks

In [3]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing '-', '.', 'N', or 'n' completely removed.
    """
    align_len = alignment_block.get_alignment_length()

    invalid_chars = {'N', 'n', '-', '.'}
    
    valid_columns = [
        i for i in range(align_len) 
        if not any(char in alignment_block[:, i] for char in invalid_chars)
    ]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    return MultipleSeqAlignment(cleaned_records)

In [4]:
xmfa_path = "../data/ecoli/coli27.xmfa"
aligned_blocks = read_xmfa_biopython(xmfa_path)

print(f"Total blocks successfully loaded: {len(aligned_blocks)}")

Total blocks successfully loaded: 765


## Load `.nwk` tree

In [5]:
clonal_tree = Phylo.read("../data/ecoli/ecoli_clonal.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

                                          ______________________________ 8
                                         |
  _______________________________________|          ____________________ 0
 |                                       |         |
 |                                       |         |                    , 21
 |                                       |_________|    ________________|
 |                                                 |   |                , 2
 |                                                 |   |                |
 |                                                 |___|                | 26
 |                                                     |
 |                                                     |   _____________ 7
 |                                                     |__|
 |                                                        |_____________ 10
 |
_|                                                                    __ 20
 |                           

In [6]:
clonal_edge = np.loadtxt("../data/ecoli/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/ecoli/clonal_node_height.csv", delimiter=",", dtype=float)

In [7]:
clonal_edge, clonal_node_height

(array([[2.80000000e+01, 7.00000000e+00, 1.98292813e-04],
        [2.80000000e+01, 8.00000000e+00, 1.98292813e-04],
        [2.90000000e+01, 1.60000000e+01, 1.02497834e-03],
        [2.90000000e+01, 1.70000000e+01, 1.02497834e-03],
        [3.00000000e+01, 5.00000000e+00, 1.89076386e-03],
        [3.00000000e+01, 6.00000000e+00, 1.89076386e-03],
        [3.10000000e+01, 2.80000000e+01, 3.36259925e-03],
        [3.10000000e+01, 3.00000000e+01, 1.67012820e-03],
        [3.20000000e+01, 1.80000000e+01, 4.39316330e-03],
        [3.20000000e+01, 1.90000000e+01, 4.39316330e-03],
        [3.30000000e+01, 3.00000000e+00, 1.43105968e-02],
        [3.30000000e+01, 4.00000000e+00, 1.43105968e-02],
        [3.40000000e+01, 2.90000000e+01, 1.39335612e-02],
        [3.40000000e+01, 3.20000000e+01, 1.05653762e-02],
        [3.50000000e+01, 2.10000000e+01, 2.13932809e-02],
        [3.50000000e+01, 2.20000000e+01, 2.13932809e-02],
        [3.60000000e+01, 2.30000000e+01, 2.64762798e-02],
        [3.600

### Count segregating sites

In [8]:
total_S = 0
total_length = 0

In [9]:
np.random.seed(100)
clonal_tree = ClonalTree(n=27)

clonal_tree.edge = clonal_edge
clonal_tree.node_height = clonal_node_height
clonal_tree.height = np.max(clonal_node_height)
clonal_tree.length = np.sum(clonal_edge[:, 2])

### Load leaf indices

In [10]:
leaf_names = np.loadtxt("../data/ecoli/tip_names.csv", delimiter=",", dtype=str)
leaf_names = np.char.strip(leaf_names, '"')
seq_names = np.array([str(i) for i in range(27)])
len(leaf_names), len(seq_names)

(27, 27)

## Compute info table and summary stats

In [11]:
def divide_evenly(n, parts):
    quotient, remainder = divmod(n, parts)
    
    # Create the list with the distributed remainder
    parts = [quotient + 1] * remainder + [quotient] * (parts - remainder)
    
    return parts

In [12]:
gene_id = []
gene_length = []
start_pos = []
end_pos = []
summary_stats_list = []
full_gap_id = []
full_gap_index = []

# change the order of the boolean matrix to match the order of the clonal tree leaves
index_map = {string_id: idx for idx, string_id in enumerate(seq_names)}
new_indices = [index_map[string_id] for string_id in leaf_names]

for i in tqdm(range(len(aligned_blocks)), desc="Processing genes"):
    # get the raw block and remove gap columns
    raw_block = aligned_blocks[i]
    clear_block = remove_gap_columns(raw_block)

    if clear_block.get_alignment_length() == 0:
        full_gap_id.append(raw_block[0].id)
        full_gap_index.append(i)
        continue

    # add the gene length and start/end positions to the info table
    if clear_block.get_alignment_length() <= 10000:
        gene_id.append(str(i))
        gene_length.append(clear_block.get_alignment_length())
        start_pos.append(raw_block[0].annotations.get('start'))
        end_pos.append(raw_block[0].annotations.get('start') + clear_block.get_alignment_length() - 1)

        # convert sequences to boolean matrix
        sequences = []
        for record in clear_block:
            seq_chars = list(str(record.seq).upper())
            sequences.append(seq_chars)

        char_matrix = np.array(sequences)
        reference_seq = char_matrix[0]
        bool_matrix_raw = (char_matrix != reference_seq)

        # change the order of the boolean matrix to match the order of the clonal tree leaves
        bool_matrix = bool_matrix_raw[new_indices]

        # compute summary statistics for the boolean matrix
        summary_stats = segment_summary_stats(clonal_tree, bool_matrix)
        summary_stats_list.append(summary_stats)

        # compute segregating sites
        has_true = bool_matrix.any(axis=0)
        has_false = ~bool_matrix.all(axis=0)
        idx_seg = np.where(has_true & has_false)[0]
        total_S += idx_seg.size
        total_length += clear_block.get_alignment_length()
    else:
        sub_segments = int(clear_block.get_alignment_length() / 10000) + 1
        sub_length = divide_evenly(clear_block.get_alignment_length(), sub_segments)
        sub_start = raw_block[0].annotations.get('start')

        # convert sequences to boolean matrix
        sequences = []
        for record in clear_block:
            seq_chars = list(str(record.seq).upper())
            sequences.append(seq_chars)

        char_matrix = np.array(sequences)
        reference_seq = char_matrix[0]
        bool_matrix_raw = (char_matrix != reference_seq)

        # change the order of the boolean matrix to match the order of the clonal tree leaves
        bool_matrix = bool_matrix_raw[new_indices]

        for j in range(sub_segments):
            gene_id.append(f"{i}_{j}")
            gene_length.append(sub_length[j])
            start_pos.append(sub_start)
            end_pos.append(sub_start + sub_length[j] - 1)
            sub_start += sub_length[j]

            # compute summary statistics for the boolean matrix
            sub_bool_matrix = bool_matrix[:, sum(sub_length[:j]):sum(sub_length[:j+1])]
            summary_stats = segment_summary_stats(clonal_tree, sub_bool_matrix)
            summary_stats_list.append(summary_stats)

            # compute segregating sites
            has_true = sub_bool_matrix.any(axis=0)
            has_false = ~sub_bool_matrix.all(axis=0)
            idx_seg = np.where(has_true & has_false)[0]
            total_S += idx_seg.size
            total_length += sub_length[j]

Processing genes:   0%|          | 0/765 [00:00<?, ?it/s]

Processing genes: 100%|██████████| 765/765 [35:21<00:00,  2.77s/it]  


In [13]:
block_info_df = pd.DataFrame({'Gene_ID': gene_id})
block_info_df['Gene_Length'] = gene_length
block_info_df['Start_pos'] = start_pos
block_info_df['End_pos'] = end_pos
block_info_df['Alignment'] = True
block_info_df

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment
0,0,1200,605790,606989,True
1,1,1724,614003,615726,True
2,2,4601,616186,620786,True
3,3,6950,606990,613939,True
4,4,3797,601974,605770,True
...,...,...,...,...,...
831,761_0,5476,1504811,1510286,True
832,761_1,5475,1510287,1515761,True
833,762,4170,1541972,1546141,True
834,763,6204,1529775,1535978,True


In [14]:
print(full_gap_id)
print(full_gap_index)

['Escherichia_coli_536.gbk/4710020-4712650']
[111]


In [15]:
print(np.min(gene_length), np.max(gene_length))

501 9988


-----

In [25]:
block_info_df.to_csv("../data/ecoli/ecoli_block_info.csv", index=False)

In [26]:
block_summary = np.array(summary_stats_list)
block_summary.shape

(836, 46)

In [27]:
np.savetxt("../data/ecoli/ecoli_block_summary_stats.csv", block_summary, delimiter=",")

theta / 2 = S / L / tree length

In [28]:
print("theta:", float(total_S / total_length / np.sum(clonal_edge[:, 2]) * 2))

theta: 0.01033768239616971
